# Aula 03: Limpeza Numérica, Regras Matemáticas e Outliers
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

Nesta terceira aula, vamos atacar o coração financeiro da planilha (`faturamento`, `margem_de_lucro` e `lucro`) e a coluna temporal (`data_da_venda`):
1. Identificar por que as colunas numéricas vieram como texto (`object`) e convertê-las com `pd.to_numeric(errors='coerce')`.
2. Detectar e corrigir **5 outliers e erros matemáticos graves** (ex: lucro de R$ 3,26 milhões em uma venda de R$ 544).
3. Recuperar quase 100 valores numéricos vazios usando a equação $\text{Lucro} = \text{Faturamento} \times \text{Margem de Lucro}$.
4. Descobrir a **segunda onda de duplicatas** que estavam disfarçadas por erros de digitação!

In [ ]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# === Replay Rápido das Aulas 01 e 02 ===
df = pd.read_excel('Campe_Supply_Corrompido.xlsx', sheet_name='Campe Supply')
df = df.drop(columns=['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('ã', 'a')
df = df.drop_duplicates().reset_index(drop=True)

categorias_oficiais = {
    'setor': ['Congelados', 'Latícinio', 'Bebidas', 'Carnes'],
    'produto': ['Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada'],
    'vendedor': ['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro'],
    'regiao': ['Sudeste', 'Nordeste', 'Sul'],
    'estado': ['Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina'],
    'cliente': ['Atacarejo', 'Varejo', 'Atacado']
}
def corrigir_palavra(valor, lista_oficial):
    if pd.isna(valor): return np.nan
    match = get_close_matches(str(valor).strip(), lista_oficial, n=1, cutoff=0.6)
    return match[0] if match else np.nan

for col, lista_valida in categorias_oficiais.items():
    df[col] = df[col].apply(lambda x: corrigir_palavra(x, lista_valida))

mapa_produto_setor = {'Queijo': 'Latícinio', 'Leite': 'Latícinio', 'Coalhada': 'Latícinio', 'Sorvete': 'Congelados', 'Lasanha': 'Congelados', 'Pizza': 'Congelados', 'Frango': 'Carnes', 'Bovina': 'Carnes', 'Suína': 'Carnes', 'Refrigerantes': 'Bebidas', 'Cervejas': 'Bebidas', 'Sucos': 'Bebidas'}
mapa_estado_regiao = {'Rio de Janeiro': 'Sudeste', 'Minas Gerais': 'Sudeste', 'São Paulo': 'Sudeste', 'Bahia': 'Nordeste', 'Rio Grande do Norte': 'Nordeste', 'Sergipe': 'Nordeste', 'Rio Grande do Sul': 'Sul', 'Paraná': 'Sul', 'Santa Catarina': 'Sul'}
df['setor'] = df['setor'].fillna(df['produto'].map(mapa_produto_setor))
df['regiao'] = df['regiao'].fillna(df['estado'].map(mapa_estado_regiao))

## 1. Por que as colunas numéricas estão como `object` (texto)?
Vamos filtrar os valores que não conseguem ser convertidos para número e ver o que aconteceu na digitação.

In [ ]:
colunas_num = ['faturamento', 'margem_de_lucro', 'lucro']

for col in colunas_num:
    serie_convertida = pd.to_numeric(df[col], errors='coerce')
    valores_corrompidos = df[df[col].notna() & serie_convertida.isna()][col]
    print(f'Exemplos de textos corrompidos em {col.upper()} (Total: {len(valores_corrompidos)}):')
    print(valores_corrompidos.head(5).tolist())
    print('-'*50)

Temos letras infiltradas no meio dos números (`"60b.72"`, `"0.a6"`, `"t6.32"`).
Se convertermos esses valores corrompidos em `NaN` usando `pd.to_numeric(..., errors='coerce')`, será que conseguimos recalcular o valor exato usando as outras duas colunas numéricas da mesma linha? Vamos testar!

In [ ]:
# Convertendo para float (valores com letras viram NaN temporariamente)
for col in colunas_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Quantidade de nulos nas colunas numéricas logo após a conversão:')
print(df[colunas_num].isna().sum())

# Olhando as estatísticas descritivas antes de limpar outliers
df[colunas_num].describe()

## 2. Caçando Outliers e Erros Matemáticos
Olhe para o `.describe()` acima:
* O `faturamento` máximo é **8.581,60**, enquanto 75% das vendas são menores que **948,20**.
* O `lucro` máximo bateu **3.260.000,00** (3,26 milhões!), enquanto a mediana é **95,24**!
* A `margem_de_lucro` mínima é **0.00**.

Vamos checar em quais linhas a conta $\text{Faturamento} \times \text{Margem} = \text{Lucro}$ **NÃO** bate!

In [ ]:
# Calculando a diferença absoluta entre (faturamento * margem) e lucro
diferenca_matematica = (df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs()

# Filtrando as linhas onde a diferença é maior que 1 centavo (0.01)
erros_matematicos = df[diferenca_matematica > 0.01]
erros_matematicos[['data_da_venda', 'produto', 'vendedor', 'faturamento', 'margem_de_lucro', 'lucro']]

Encontramos exatamente as **5 linhas problemáticas** da base!
1. **Linhas 304 e 1304:** O `lucro` está maior que o próprio `faturamento` (erro de multiplicação/casas decimais). Basta recalcular `lucro = faturamento * margem_de_lucro`.
2. **Linhas 975 e 1426:** A `margem_de_lucro` foi digitada como `0.00`, mas temos o `faturamento` e o `lucro` corretos. Basta recalcular `margem_de_lucro = lucro / faturamento`.
3. **Linha 572:** O `faturamento` foi multiplicado por 10 sem querer (`8581.60` em vez de `858.16`), pois `858.16 * 0.06 = 51.4896`. Basta recalcular `faturamento = lucro / margem_de_lucro`.

In [ ]:
# 1. Corrigindo lucros absurdos (onde lucro > faturamento)
mask_lucro_errado = df['lucro'] > df['faturamento']
df.loc[mask_lucro_errado, 'lucro'] = df.loc[mask_lucro_errado, 'faturamento'] * df.loc[mask_lucro_errado, 'margem_de_lucro']

# 2. Corrigindo margens zeradas onde há lucro positivo
mask_margem_zero = (df['margem_de_lucro'] == 0) & (df['lucro'] > 0)
df.loc[mask_margem_zero, 'margem_de_lucro'] = df.loc[mask_margem_zero, 'lucro'] / df.loc[mask_margem_zero, 'faturamento']

# 3. Corrigindo o faturamento discrepante (onde a conta ainda não bate)
mask_fat_errado = (df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs() > 0.01
df.loc[mask_fat_errado, 'faturamento'] = df.loc[mask_fat_errado, 'lucro'] / df.loc[mask_fat_errado, 'margem_de_lucro']

print('Linhas com erro matemático restantes:', ((df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs() > 0.01).sum())

## 3. Recuperando os Nulos Numéricos com Matemática
Agora que as colunas são confiáveis, podemos preencher os valores `NaN` sempre que uma linha tiver 2 das 3 informações financeiras:

In [ ]:
# Preenchendo nulos usando a relação matemática entre as 3 colunas
df['lucro'] = df['lucro'].fillna(df['faturamento'] * df['margem_de_lucro'])
df['faturamento'] = df['faturamento'].fillna(df['lucro'] / df['margem_de_lucro'])
df['margem_de_lucro'] = df['margem_de_lucro'].fillna(df['lucro'] / df['faturamento'])

# Arredondando para 2 casas decimais
df['faturamento'] = df['faturamento'].round(2)
df['margem_de_lucro'] = df['margem_de_lucro'].round(2)
df['lucro'] = df['lucro'].round(2)

print('Nulos numéricos APÓS a recuperação matemática:')
print(df[colunas_num].isna().sum())
df[colunas_num].describe()

## 4. A Segunda Onda de Duplicatas e Engenharia de Datas
Lembra que na Aula 02 corrigimos erros de digitação como `"Vacessa" -> "Vanessa"` e agora na Aula 03 recuperamos números que estavam como `"60b.72" -> 604.72`?
Muitas linhas que eram duplicatas estavam **escondidas** atrás desses erros de digitação! Vamos verificar quantas duplicatas surgiram agora que a tabela está limpa:

In [ ]:
print('Novas duplicatas reveladas após a limpeza de textos e números:', df.duplicated().sum())

# Removendo as novas duplicatas
df = df.drop_duplicates().reset_index(drop=True)
print('Formato da base após 2ª desduplicação:', df.shape)

In [ ]:
# Criando colunas temporais a partir de data_da_venda para usar nas análises da Aula 04 e 05
df['mes'] = df['data_da_venda'].dt.month
df['trimestre'] = df['data_da_venda'].dt.quarter

# Salvando a base limpa para usar nas próximas aulas
df.to_excel('Campe_Supply_Limpo.xlsx', index=False)
print('Arquivo Campe_Supply_Limpo.xlsx exportado com sucesso!')